In [ ]:
#ｘ−ｙ平面の密度map作成用コード（jeans_3d-test14の可視化に使ったやつ）

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm
from PIL import Image
import matplotlib.animation as animation

#密度画像（x-y平面密度）

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

# 計算領域・球パラメータ
Lbox = 12.0
xc, yc = 6.0, 6.0
z_slice = 6.0          # 箱中心
Rsphere = 5.0

# 全 block の VTK ファイルを自動取得（block0〜blockXX）
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

# ブロック番号ごとにまとめる（x-y平面密度）

block_dict = {}
for f in vtk_files:
    basename = os.path.basename(f)
    block_id = int(basename.split('.')[1].replace('block',''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内のみで vmin / vmax を決定
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):
    points_all = []
    dens_all = []

    for f in step_files:
        grid = pv.read(f)
        xy_slice = grid.slice(normal='z', origin=(0,0,z_slice))
        centers = xy_slice.cell_centers().points
        dens = xy_slice['dens']

        points_all.append(centers)
        dens_all.append(dens)

    points_all = np.vstack(points_all)
    dens_all = np.hstack(dens_all)

    r = np.sqrt((points_all[:,0]-xc)**2 + (points_all[:,1]-yc)**2)
    mask_inside = r <= Rsphere
    dens_inside_all.append(dens_all[mask_inside])

dens_inside_all = np.concatenate(dens_inside_all)

# 極端値を除外
vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

print(f"Density scale (inside sphere): vmin={vmin}, vmax={vmax}")

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    points_all = []
    dens_all = []

    for f in step_files:
        grid = pv.read(f)
        xy_slice = grid.slice(normal='z', origin=(0,0,z_slice))
        centers = xy_slice.cell_centers().points
        dens = xy_slice['dens']

        points_all.append(centers)
        dens_all.append(dens)

    points_all = np.vstack(points_all)
    dens_all = np.hstack(dens_all)

    xs_unique = np.unique(points_all[:,0])
    ys_unique = np.unique(points_all[:,1])
    nx, ny = len(xs_unique), len(ys_unique)

    dens_2d = np.full((ny, nx), np.nan)

    for i, x in enumerate(xs_unique):
        for j, y in enumerate(ys_unique):
            mask = (np.isclose(points_all[:,0], x)) & \
                   (np.isclose(points_all[:,1], y))
            if np.any(mask):
                dens_2d[j, i] = dens_all[mask][0]

    # 球内・球外マスク
    X, Y = np.meshgrid(xs_unique, ys_unique)
    r = np.sqrt((X-xc)**2 + (Y-yc)**2)

    dens_in  = np.ma.masked_where(r > Rsphere, dens_2d)
    dens_out = np.ma.masked_where(r <= Rsphere, dens_2d)

    # --------------------------------------------------------
    # 描画
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_aspect('equal')

    # 球外：背景（単色）
    ax.pcolormesh(xs_unique, ys_unique, dens_out,
                  cmap='Greys', vmin=0, vmax=1)

    # 球内：密度勾配（対数）
    im = ax.pcolormesh(xs_unique, ys_unique, dens_in,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            yc + Rsphere*np.sin(theta),
            'c--', lw=1.2)

    ax.set_xlim(0, Lbox)
    ax.set_ylim(0, Lbox)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    filename = os.path.basename(step_files[0])
    num = filename.split('.')[-2]
    ax.set_title(f"file={filename}")

    fig.colorbar(im, ax=ax, label="Density (inside sphere)")

    png_file = os.path.join(output_dir, f"dens_{num}.png")
    fig.savefig(png_file, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved {png_file}")

In [ ]:
#ｘ−ｙ平面の密度map作成用コード（jeans_3d-test16の可視化に使ったやつ）
#球外へのmassの漏れ出しが視覚的にわかりやすい

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm
from PIL import Image

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

Lbox = 8.0
xc, yc = 4.0, 4.0
z_slice = 4.0
Rsphere = 1.25
rho_bg = 1e-3   # 入力ファイルと合わせる

# ============================================================
# VTK ファイル整理（block ごと）
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内密度から vmin / vmax を決定
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:,0])
    ys = np.unique(pts_all[:,1])

    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iy = np.where(ys == p[1])[0][0]
        dens_2d[iy, ix] = d

    X, Y = np.meshgrid(xs, ys)
    r2d  = np.sqrt((X-xc)**2 + (Y-yc)**2)

    dens_inside_all.append(dens_2d[r2d <= Rsphere])

dens_inside_all = np.concatenate(dens_inside_all)

vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

print(f"[INFO] Density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:,0])
    ys = np.unique(pts_all[:,1])

    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iy = np.where(ys == p[1])[0][0]
        dens_2d[iy, ix] = d

    X, Y = np.meshgrid(xs, ys)
    r2d  = np.sqrt((X-xc)**2 + (Y-yc)**2)

    fig, ax = plt.subplots(figsize=(6,6))
    ax.set_aspect('equal')

    # ===============================
    # ① 背景：全領域を linear 表示
    # ===============================
    ax.pcolormesh(xs, ys, dens_2d,
                  cmap='Greys',
                  vmin=rho_bg,
                  vmax=vmin)

    # ===============================
    # ② 球内：LogNorm で上書き
    # ===============================
    mask_sphere = r2d <= Rsphere
    dens_sphere = np.ma.masked_where(~mask_sphere, dens_2d)

    im = ax.pcolormesh(xs, ys, dens_sphere,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            yc + Rsphere*np.sin(theta),
            'c--', lw=1.2)

    ax.set_xlim(0, Lbox)
    ax.set_ylim(0, Lbox)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    fname = os.path.basename(step_files[0])
    num   = fname.split('.')[-2]
    ax.set_title(f"z={z_slice}, γ=2.0, β=0.02")

    fig.colorbar(im, ax=ax, label="Density")

    png = os.path.join(output_dir, f"dens_{num}.png")
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved {png}")

In [ ]:
#可視化のために球外に漏れ出た密度をいじって色付けしている。球外の色付けは実際に漏れ出た密度の絶対値ではないので注意
#スパイラルアームなどが立つとき、より見やすい可視化コード（jeans_3d-test24を可視化したときに使った）
#x-y平面用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
from PIL import Image
from matplotlib.colors import LogNorm, ListedColormap

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

Lbox = 12.0
xc, yc = 6.0, 6.0
z_slice = 6.0
Rsphere = 5.0

rho_bg   = 1e-15            # 初期背景密度
rho_leak = 5.0 * rho_bg    # 漏れ出た質量の判定閾値（固定）

# 漏れ出た質量用：単色・強調ブルー
leak_cmap = ListedColormap(["deepskyblue"])

# ============================================================
# VTK ファイル整理（block ごと）
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内密度から vmin / vmax を決定（全ステップ共通）
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:, 0])
    ys = np.unique(pts_all[:, 1])

    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iy = np.where(ys == p[1])[0][0]
        dens_2d[iy, ix] = d

    X, Y = np.meshgrid(xs, ys)
    r2d  = np.sqrt((X - xc)**2 + (Y - yc)**2)

    dens_inside_all.append(dens_2d[r2d <= Rsphere])

dens_inside_all = np.concatenate(dens_inside_all)

vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

print(f"[INFO] Sphere density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:, 0])
    ys = np.unique(pts_all[:, 1])

    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iy = np.where(ys == p[1])[0][0]
        dens_2d[iy, ix] = d

    X, Y = np.meshgrid(xs, ys)
    r2d  = np.sqrt((X - xc)**2 + (Y - yc)**2)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')

    # ========================================================
    # ① 球外背景（薄グレー）
    # ========================================================
    mask_bg = (r2d > Rsphere) & (dens_2d <= rho_leak)
    dens_bg = np.ma.masked_where(~mask_bg, dens_2d)

    ax.pcolormesh(xs, ys, dens_bg,
                  cmap='Greys',
                  vmin=rho_bg,
                  vmax=rho_leak,
                  alpha=0.35)

    # ========================================================
    # ② 漏れ出た質量（単色・濃いブルー）
    # ========================================================
    mask_leak = (r2d > Rsphere) & (dens_2d > rho_leak)
    dens_leak = np.ma.masked_where(~mask_leak, dens_2d)

    ax.pcolormesh(xs, ys, dens_leak,
                  cmap=leak_cmap,
                  alpha=0.45)

    # 境界を等高線で強調（スパイラル検出用）
    ax.contour(X, Y, dens_2d,
               levels=[rho_leak],
               colors='dodgerblue',
               linewidths=1.5)

    # ========================================================
    # ③ 球内（LogNorm）
    # ========================================================
    mask_sphere = r2d <= Rsphere
    dens_sphere = np.ma.masked_where(~mask_sphere, dens_2d)

    im = ax.pcolormesh(xs, ys, dens_sphere,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            yc + Rsphere*np.sin(theta),
            'c--', lw=1.2)

    ax.set_xlim(0, Lbox)
    ax.set_ylim(0, Lbox)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    fname = os.path.basename(step_files[0])
    num   = fname.split('.')[-2]
    ax.set_title(f"z={z_slice}, γ=1.2, β=0.02")

    fig.colorbar(im, ax=ax, label="Density (sphere)")

    png = os.path.join(output_dir, f"dens_{num}.png")
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved {png}")


In [ ]:
#可視化のために球外に漏れ出た密度をいじって色付けしている。球外の色付けは実際に漏れ出た密度の絶対値ではないので注意
#スパイラルアームなどが立つとき、より見やすい可視化コード（jeans_3d-test24を可視化したときに使った）
#x-z平面用



import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
import natsort
from matplotlib.colors import LogNorm, ListedColormap

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xz_density_maps"
os.makedirs(output_dir, exist_ok=True)

Lbox = 12.0
xc, yc, zc = 6.0, 6.0, 6.0
y_slice = 6.0          # ★ y でスライス
Rsphere = 5.0

rho_bg   = 1e-15
rho_leak = 5.0 * rho_bg

# 漏れ出た質量用：単色ブルー
leak_cmap = ListedColormap(["deepskyblue"])

# ============================================================
# VTK ファイル整理（block ごと）
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.startswith("Jeans.block") and f.endswith(".vtk")
])

block_dict = {}
for f in vtk_files:
    block_id = int(os.path.basename(f).split('.')[1].replace('block', ''))
    block_dict.setdefault(block_id, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]

# ============================================================
# 球内密度から vmin / vmax を決定（全ステップ共通）
# ============================================================
dens_inside_all = []

for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='y', origin=(0.0, y_slice, 0.0))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:, 0])
    zs = np.unique(pts_all[:, 2])

    dens_2d = np.full((len(zs), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iz = np.where(zs == p[2])[0][0]
        dens_2d[iz, ix] = d

    X, Z = np.meshgrid(xs, zs)
    r2d  = np.sqrt((X - xc)**2 + (Z - zc)**2)

    dens_inside_all.append(dens_2d[r2d <= Rsphere])

dens_inside_all = np.concatenate(dens_inside_all)
vmin = np.percentile(dens_inside_all, 5)
vmax = np.percentile(dens_inside_all, 95)

print(f"[INFO] Sphere density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")

# ============================================================
# 各タイムステップ描画
# ============================================================
for step_files in zip(*all_blocks_sorted):

    pts_all, dens_all = [], []

    for f in step_files:
        grid = pv.read(f)
        slc  = grid.slice(normal='y', origin=(0.0, y_slice, 0.0))
        pts_all.append(slc.cell_centers().points)
        dens_all.append(slc['dens'])

    pts_all  = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)

    xs = np.unique(pts_all[:, 0])
    zs = np.unique(pts_all[:, 2])

    dens_2d = np.full((len(zs), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(xs == p[0])[0][0]
        iz = np.where(zs == p[2])[0][0]
        dens_2d[iz, ix] = d

    X, Z = np.meshgrid(xs, zs)
    r2d  = np.sqrt((X - xc)**2 + (Z - zc)**2)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')

    # ========================================================
    # ① 球外背景（薄グレー）
    # ========================================================
    mask_bg = (r2d > Rsphere) & (dens_2d <= rho_leak)
    dens_bg = np.ma.masked_where(~mask_bg, dens_2d)

    ax.pcolormesh(xs, zs, dens_bg,
                  cmap='Greys',
                  vmin=rho_bg,
                  vmax=rho_leak,
                  alpha=0.35)

    # ========================================================
    # ② 漏れ出た質量（ブルー）
    # ========================================================
    mask_leak = (r2d > Rsphere) & (dens_2d > rho_leak)
    dens_leak = np.ma.masked_where(~mask_leak, dens_2d)

    ax.pcolormesh(xs, zs, dens_leak,
                  cmap=leak_cmap,
                  alpha=0.45)

    ax.contour(X, Z, dens_2d,
               levels=[rho_leak],
               colors='dodgerblue',
               linewidths=1.5)

    # ========================================================
    # ③ 球内（LogNorm）
    # ========================================================
    mask_sphere = r2d <= Rsphere
    dens_sphere = np.ma.masked_where(~mask_sphere, dens_2d)

    im = ax.pcolormesh(xs, zs, dens_sphere,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax))

    # 球境界
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(xc + Rsphere*np.cos(theta),
            zc + Rsphere*np.sin(theta),
            'c--', lw=1.2)

    ax.set_xlim(0, Lbox)
    ax.set_ylim(0, Lbox)
    ax.set_xlabel("X")
    ax.set_ylabel("Z")

    fname = os.path.basename(step_files[0])
    num   = fname.split('.')[-2]
    ax.set_title(f"y={y_slice}, γ=1.2, β=0.02")

    fig.colorbar(im, ax=ax, label="Density (sphere)")

    png = os.path.join(output_dir, f"dens_xz_{num}.png")
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved {png}")

In [ ]:
#Toyouchi-test4を可視化するとき使った
#Toyouchi+23再現ではこれを使うと良い
#x-y平面用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/your_result_directory")  # ★要修正★
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

# 計算領域の設定（一辺448の立方体、中心が原点）
Lbox = 448.0
# 座標範囲: -224 から 224 まで
x_min, x_max = -Lbox/2, Lbox/2  # -224 から 224
y_min, y_max = -Lbox/2, Lbox/2
z_slice = 0.0  # 真ん中で切断（原点を通る面）

# 中心は原点
xc, yc = 0.0, 0.0

# プロットの解像度を上げるための設定
dpi = 300  # 高解像度で保存
figsize = (12, 10)  # 図のサイズを大きく

# ============================================================
# デバッグ: VTKファイルの確認
# ============================================================
print(f"[DEBUG] Looking for VTK files in: {vtk_dir}")
print(f"[DEBUG] Directory exists: {os.path.exists(vtk_dir)}")

if os.path.exists(vtk_dir):
    all_files = os.listdir(vtk_dir)
    vtk_files_candidates = [f for f in all_files if f.endswith(".vtk")]
    print(f"[DEBUG] Total .vtk files found: {len(vtk_files_candidates)}")
    if len(vtk_files_candidates) > 0:
        print(f"[DEBUG] First 5 VTK files: {vtk_files_candidates[:5]}")
else:
    print(f"[ERROR] Directory does not exist: {vtk_dir}")
    exit()

# ============================================================
# VTK ファイル整理（block ごと）
# ============================================================
vtk_files = natsort.natsorted([
    os.path.join(vtk_dir, f)
    for f in os.listdir(vtk_dir)
    if f.endswith(".vtk")
])

if not vtk_files:
    print("[ERROR] VTK files not found!")
    exit()

print(f"[INFO] Found {len(vtk_files)} VTK files")

# blockごとにグループ化
block_dict = {}
for f in vtk_files:
    try:
        basename = os.path.basename(f)
        # 様々なパターンに対応
        if '.block' in basename:
            if '.block_' in basename:
                block_id = int(basename.split('.block_')[1].split('.')[0])
            else:
                block_id = int(basename.split('.block')[1].split('.')[0])
        else:
            import re
            numbers = re.findall(r'\d+', basename)
            if numbers:
                block_id = int(numbers[0])
            else:
                block_id = 0
        block_dict.setdefault(block_id, []).append(f)
    except Exception as e:
        print(f"[WARNING] Could not parse block ID from {f}: {e}")
        block_dict.setdefault(0, []).append(f)

all_blocks_sorted = [block_dict[b] for b in sorted(block_dict.keys())]
print(f"[INFO] Number of blocks: {len(all_blocks_sorted)}")

# タイムステップ数を確認
n_timesteps = min(len(block_files) for block_files in all_blocks_sorted)
print(f"[INFO] Number of timesteps: {n_timesteps}")

if n_timesteps == 0:
    print("[ERROR] No timesteps found!")
    exit()

# ============================================================
# 全ステップの密度からvmin/vmaxを決定
# ============================================================
all_densities = []
print("[INFO] Collecting density data for colormap scaling...")

sample_steps = min(10, n_timesteps)

for step_idx in range(sample_steps):
    dens_list = []
    for block_files in all_blocks_sorted:
        if step_idx < len(block_files):
            f = block_files[step_idx]
            try:
                grid = pv.read(f)
                slc = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
                dens_list.append(slc['dens'])
            except Exception as e:
                print(f"[WARNING] Could not read file {f}: {e}")
    
    if dens_list:
        dens_all = np.hstack(dens_list)
        all_densities.append(dens_all)

if all_densities:
    all_densities = np.concatenate(all_densities)
    positive_dens = all_densities[all_densities > 0]
    
    if len(positive_dens) > 0:
        # より細かい構造を見るためにパーセンタイルを調整
        vmin = np.percentile(positive_dens, 2)   # 下限を2%に
        vmax = np.percentile(positive_dens, 98)  # 上限を98%に
        print(f"[INFO] Density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")
    else:
        vmin, vmax = 1e-10, 1e-5
        print("[WARNING] No positive densities found, using default scale")
else:
    print("[ERROR] No density data could be read!")
    exit()

# ============================================================
# 各タイムステップ描画（高解像度版）
# ============================================================
print("[INFO] Generating high-resolution density maps...")

for step_idx in range(n_timesteps):
    
    # 全ブロックの点と密度を収集
    pts_all = []
    dens_all = []
    
    for block_files in all_blocks_sorted:
        if step_idx < len(block_files):
            f = block_files[step_idx]
            try:
                grid = pv.read(f)
                slc = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
                pts_all.append(slc.cell_centers().points)
                dens_all.append(slc['dens'])
            except Exception as e:
                continue
    
    if not pts_all:
        print(f"[WARNING] No data for timestep {step_idx}, skipping...")
        continue
    
    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)
    
    # グリッド座標の取得（ソートして一意な値を取得）
    xs = np.sort(np.unique(pts_all[:, 0]))
    ys = np.sort(np.unique(pts_all[:, 1]))
    
    # 2D配列にマッピング（より滑らかな表示のために補間を有効化）
    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(np.isclose(xs, p[0], rtol=1e-6, atol=1e-8))[0][0]
        iy = np.where(np.isclose(ys, p[1], rtol=1e-6, atol=1e-8))[0][0]
        dens_2d[iy, ix] = d
    
    # より滑らかな表示のために線形補間を適用
    from scipy.interpolate import griddata
    # 元のグリッド点
    points_2d = np.column_stack([pts_all[:, 0], pts_all[:, 1]])
    # より細かいグリッドを作成（解像度を2倍に）
    xs_fine = np.linspace(x_min, x_max, len(xs) * 2)
    ys_fine = np.linspace(y_min, y_max, len(ys) * 2)
    X_fine, Y_fine = np.meshgrid(xs_fine, ys_fine)
    # 線形補間
    dens_2d_fine = griddata(points_2d, dens_all, (X_fine, Y_fine), method='linear')
    
    # プロット
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.set_aspect('equal')
    
    # 密度のヒートマップ（高解像度、対数スケール）
    im = ax.pcolormesh(X_fine, Y_fine, dens_2d_fine,
                       cmap='inferno',
                       norm=LogNorm(vmin=vmin, vmax=vmax),
                       shading='auto',
                       rasterized=True)  # ファイルサイズ最適化
    
    # 中心に原点マーク
    ax.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, 
            label='Center (Origin)', zorder=10)
    
    # 半径を表示するための円を追加（オプション）
    radii = [50, 100, 150, 200]
    for r in radii:
        if r <= Lbox/2:
            theta = np.linspace(0, 2*np.pi, 500)
            ax.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                   'w--', alpha=0.3, linewidth=0.5, zorder=5)
            ax.text(xc + r*0.7, yc + r*0.7, f'r={r}', 
                   color='white', fontsize=8, alpha=0.5, zorder=5)
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("X", fontsize=14)
    ax.set_ylabel("Y", fontsize=14)
    
    ax.set_title(f"Density Distribution (x-y plane at z=0)\n"
                 f"ρ ∝ r^-1.75, Domain: [{x_min:.0f}, {x_max:.0f}]", 
                 fontsize=14, fontweight='bold')
    
    # カラーバー（より詳細な表示）
    cbar = fig.colorbar(im, ax=ax, label="Density", extend='both')
    cbar.set_label("Density", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    
    # グリッドを追加（構造を明確に）
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    
    ax.legend(loc='upper right', framealpha=0.7)
    
    plt.tight_layout()
    
    # 保存（高解像度）
    png = os.path.join(output_dir, f"density_xy_timestep_{step_idx:04d}.png")
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    if (step_idx + 1) % 5 == 0:
        print(f"[INFO] Processed {step_idx + 1}/{n_timesteps} time steps...")

print(f"[INFO] All high-resolution density maps saved to {output_dir}")
print(f"[INFO] Image size: {figsize[0]*dpi:.0f} x {figsize[1]*dpi:.0f} pixels")

In [ ]:
#Toyouchi-test4を可視化するのに使った
#半径100以内を拡大表示して全体図の右側に並べて表示する(AMR_OFFバージョン)
#x-y平面用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from collections import defaultdict

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

# 計算領域の設定
Lbox = 448.0
x_min, x_max = -Lbox/2, Lbox/2
y_min, y_max = -Lbox/2, Lbox/2
z_slice = 0.0

xc, yc = 0.0, 0.0
high_res_radius = 100.0
dpi = 200
figsize = (16, 8)

# ============================================================
# VTK ファイル整理（改良版）
# ============================================================
print("[INFO] Organizing VTK files...")

timestep_dict = defaultdict(list)
all_timesteps = set()

for f in os.listdir(vtk_dir):
    if f.endswith(".vtk") and f.startswith("Jeans.block"):
        # ファイル名: Jeans.blockXXX.out2.YYYYY.vtk
        try:
            # タイムステップ番号を抽出（YYYYYの部分）
            # .out2. の後ろから .vtk の前まで
            timestep_str = f.split('.out2.')[1].split('.')[0]
            timestep = int(timestep_str)
            timestep_dict[timestep].append(os.path.join(vtk_dir, f))
            all_timesteps.add(timestep)
        except (IndexError, ValueError) as e:
            print(f"[WARNING] Could not parse timestep from: {f}")
            continue

# タイムステップをソート
timesteps = sorted(all_timesteps)
print(f"[INFO] Found {len(timesteps)} timesteps")
print(f"[INFO] Timestep range: {timesteps[0]} - {timesteps[-1]}")
print(f"[INFO] Files per timestep: {len(timestep_dict[timesteps[0]])}")

# 全タイムステップのリストを確認
print(f"[INFO] First 10 timesteps: {timesteps[:10]}")
print(f"[INFO] Last 10 timesteps: {timesteps[-10:]}")

# ============================================================
# 密度スケールの決定
# ============================================================
print("[INFO] Determining density scale...")

# 全タイムステップから代表的なものをサンプリング
sample_indices = list(range(0, len(timesteps), len(timesteps)//10))[:10]
sample_timesteps = [timesteps[i] for i in sample_indices]
print(f"[INFO] Sampling timesteps: {sample_timesteps}")

all_densities = []

for ts in sample_timesteps:
    dens_list = []
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            slc = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
            dens_list.append(slc['dens'])
        except Exception as e:
            continue
    if dens_list:
        all_densities.append(np.hstack(dens_list))
        print(f"[INFO] Sampled timestep {ts}: {len(dens_list[-1])} points")

if all_densities:
    all_densities = np.concatenate(all_densities)
    positive_dens = all_densities[all_densities > 0]
    
    if len(positive_dens) > 0:
        vmin = np.percentile(positive_dens, 1)
        vmax = np.percentile(positive_dens, 99)
        print(f"[INFO] Density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")
        print(f"[INFO] Density range: {positive_dens.min():.3e} - {positive_dens.max():.3e}")
    else:
        vmin, vmax = 1e-6, 1e-2
        print(f"[INFO] Using default density scale: {vmin:.3e} - {vmax:.3e}")
else:
    vmin, vmax = 1e-6, 1e-2
    print(f"[INFO] Using default density scale: {vmin:.3e} - {vmax:.3e}")

# ============================================================
# 各タイムステップ描画（全タイムステップ）
# ============================================================
print("[INFO] Generating density maps for all timesteps...")

# 全タイムステップをプロット
timesteps_to_plot = timesteps  # 全タイムステップ

for ts_idx, ts in enumerate(timesteps_to_plot):
    print(f"[INFO] Processing timestep {ts} ({ts_idx+1}/{len(timesteps_to_plot)})")
    
    # 全ブロックの点と密度を収集
    pts_all = []
    dens_all = []
    
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            slc = grid.slice(normal='z', origin=(0.0, 0.0, z_slice))
            # cell_centersを使うとセル中心の座標が得られる
            pts_all.append(slc.cell_centers().points)
            dens_all.append(slc['dens'])
        except Exception as e:
            continue
    
    if not pts_all:
        print(f"[WARNING] No data for timestep {ts}, skipping...")
        continue
    
    pts_all = np.vstack(pts_all)
    dens_all = np.hstack(dens_all)
    
    # デバッグ: 最初の数ステップでデータ確認
    if ts_idx < 3:
        print(f"[DEBUG] Timestep {ts}: {len(pts_all)} points")
        print(f"[DEBUG] X range: {pts_all[:,0].min():.1f} - {pts_all[:,0].max():.1f}")
        print(f"[DEBUG] Y range: {pts_all[:,1].min():.1f} - {pts_all[:,1].max():.1f}")
        print(f"[DEBUG] Density range: {dens_all.min():.3e} - {dens_all.max():.3e}")
    
    # 内側領域のデータ抽出
    r_all = np.sqrt(pts_all[:, 0]**2 + pts_all[:, 1]**2)
    mask_inner = r_all <= high_res_radius
    
    # グリッド生成
    xs = np.sort(np.unique(pts_all[:, 0]))
    ys = np.sort(np.unique(pts_all[:, 1]))
    
    # 2D配列に変換
    dens_2d = np.full((len(ys), len(xs)), np.nan)
    for p, d in zip(pts_all, dens_all):
        ix = np.where(np.isclose(xs, p[0], rtol=1e-5, atol=1e-8))[0]
        iy = np.where(np.isclose(ys, p[1], rtol=1e-5, atol=1e-8))[0]
        if len(ix) > 0 and len(iy) > 0:
            dens_2d[iy[0], ix[0]] = d
    
    X, Y = np.meshgrid(xs, ys)
    
    # 内側領域用の高解像度グリッド
    if mask_inner.sum() > 0:
        pts_inner = pts_all[mask_inner]
        dens_inner = dens_all[mask_inner]
        xs_inner = np.sort(np.unique(pts_inner[:, 0]))
        ys_inner = np.sort(np.unique(pts_inner[:, 1]))
        
        if len(xs_inner) > 1 and len(ys_inner) > 1:
            xs_fine = np.linspace(xs_inner.min(), xs_inner.max(), len(xs_inner) * 2)
            ys_fine = np.linspace(ys_inner.min(), ys_inner.max(), len(ys_inner) * 2)
            X_inner, Y_inner = np.meshgrid(xs_fine, ys_fine)
            
            points_2d = np.column_stack([pts_inner[:, 0], pts_inner[:, 1]])
            dens_inner_fine = griddata(points_2d, dens_inner, (X_inner, Y_inner), method='linear')
        else:
            X_inner, Y_inner, dens_inner_fine = None, None, None
    else:
        X_inner, Y_inner, dens_inner_fine = None, None, None
    
    # ====================================================
    # プロット
    # ====================================================
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 3, figure=fig, width_ratios=[4, 4, 0.3], wspace=0.3)
    
    # 全体図
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_aspect('equal')
    
    im1 = ax1.pcolormesh(X, Y, dens_2d,
                         cmap='inferno',
                         norm=LogNorm(vmin=vmin, vmax=vmax),
                         shading='auto')
    
    # 高解像度領域を示す四角形
    rect = patches.Rectangle(
        (-high_res_radius, -high_res_radius),
        2*high_res_radius, 2*high_res_radius,
        linewidth=2, edgecolor='cyan', facecolor='none',
        linestyle='--', label=f'Zoom region (r={high_res_radius})'
    )
    ax1.add_patch(rect)
    
    # 中心マーク
    ax1.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
    
    # 半径の円
    for r in [50, 100, 150, 200]:
        if r <= Lbox/2:
            theta = np.linspace(0, 2*np.pi, 200)
            ax1.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.3, linewidth=0.8)
            if r in [100, 200]:
                ax1.text(xc + r*0.7, yc + r*0.7, f'r={r}', 
                        color='white', fontsize=8, alpha=0.6)
    
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    ax1.set_xlabel("X", fontsize=12)
    ax1.set_ylabel("Y", fontsize=12)
    ax1.set_title(f"Full Domain (448³)", fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax1.legend(loc='upper right', framealpha=0.7, fontsize=8)
    
    # 拡大図
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_aspect('equal')
    
    if X_inner is not None and dens_inner_fine is not None:
        im2 = ax2.pcolormesh(X_inner, Y_inner, dens_inner_fine,
                             cmap='inferno',
                             norm=LogNorm(vmin=vmin, vmax=vmax),
                             shading='auto')
        
        ax2.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
        
        for r in [20, 40, 60, 80, 100]:
            theta = np.linspace(0, 2*np.pi, 200)
            ax2.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.5, linewidth=0.8)
            ax2.text(xc + r*0.7, yc + r*0.7, f'r={r}', 
                    color='white', fontsize=9, alpha=0.7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
        
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
        ax2.set_xlabel("X", fontsize=12)
        ax2.set_ylabel("Y", fontsize=12)
        ax2.set_title(f"Zoomed Region (r ≤ {high_res_radius})\nHigh Resolution", 
                     fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
        ax2.legend(loc='upper right', framealpha=0.7, fontsize=8)
    else:
        ax2.text(0.5, 0.5, 'No data in inner region', 
                transform=ax2.transAxes, ha='center', va='center', fontsize=12)
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
    
    # カラーバー
    cax = fig.add_subplot(gs[0, 2])
    cbar = fig.colorbar(im1, cax=cax, label="Density", extend='both')
    cbar.set_label("Density", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    
    # タイトル
    fig.suptitle(f"Density Distribution (x-y plane at z=0)\n"
                 f"ρ ∝ r^-1.75 | Timestep: {ts:05d} AMR_OFF", 
                 fontsize=14, fontweight='bold', y=0.98)
    
    plt.subplots_adjust(top=0.92, bottom=0.08, left=0.05, right=0.95, wspace=0.25)
    
    # 保存
    png = os.path.join(output_dir, f"density_xy_timestep_{ts:05d}_zoom.png")
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    if (ts_idx + 1) % 10 == 0:
        print(f"[INFO] Processed {ts_idx + 1}/{len(timesteps_to_plot)} timesteps")

print(f"[INFO] All density maps saved to {output_dir}")
print(f"[INFO] Total images: {len(timesteps_to_plot)}")

In [ ]:
#Toyouchi-test5を可視化するのに使った
#半径100以内を拡大表示して全体図の右側に並べて表示する(AMR_ONバージョン)
#x-y平面用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
import natsort
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from collections import defaultdict

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")  # AMRあり
output_dir = "./xy_density_maps"
os.makedirs(output_dir, exist_ok=True)

# 計算領域の設定
Lbox = 448.0
x_min, x_max = -Lbox/2, Lbox/2
y_min, y_max = -Lbox/2, Lbox/2
z_slice = 0.0

xc, yc = 0.0, 0.0
high_res_radius = 100.0
dpi = 200
figsize = (16, 8)

# ============================================================
# VTK ファイル整理
# ============================================================
print("[INFO] Organizing VTK files...")

timestep_dict = defaultdict(list)
all_timesteps = set()

for f in os.listdir(vtk_dir):
    if f.endswith(".vtk") and f.startswith("Jeans.block"):
        try:
            timestep_str = f.split('.out2.')[1].split('.')[0]
            timestep = int(timestep_str)
            timestep_dict[timestep].append(os.path.join(vtk_dir, f))
            all_timesteps.add(timestep)
        except (IndexError, ValueError):
            continue

timesteps = sorted(all_timesteps)
print(f"[INFO] Found {len(timesteps)} timesteps")
print(f"[INFO] Timestep range: {timesteps[0]} - {timesteps[-1]}")
print(f"[INFO] Files per timestep: {len(timestep_dict[timesteps[0]])}")

# ============================================================
# 密度スケールの決定
# ============================================================
print("[INFO] Determining density scale...")

# 全タイムステップから均等にサンプリング（最大10ステップ）
sample_indices = np.linspace(0, len(timesteps)-1, min(10, len(timesteps))).astype(int)
sample_timesteps = [timesteps[i] for i in sample_indices]
print(f"[INFO] Sampling timesteps: {sample_timesteps}")

all_densities = []

for ts in sample_timesteps:
    dens_list = []
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            dens_list.append(grid['dens'])
        except Exception as e:
            continue
    if dens_list:
        all_densities.append(np.hstack(dens_list))

if all_densities:
    all_densities = np.concatenate(all_densities)
    positive_dens = all_densities[all_densities > 0]
    
    if len(positive_dens) > 0:
        vmin = np.percentile(positive_dens, 1)
        vmax = np.percentile(positive_dens, 99)
        print(f"[INFO] Density scale: vmin={vmin:.3e}, vmax={vmax:.3e}")
        print(f"[INFO] Density range: {positive_dens.min():.3e} - {positive_dens.max():.3e}")
    else:
        vmin, vmax = 1e-6, 1e-2
        print(f"[INFO] Using default density scale: {vmin:.3e} - {vmax:.3e}")
else:
    vmin, vmax = 1e-6, 1e-2
    print(f"[INFO] Using default density scale: {vmin:.3e} - {vmax:.3e}")

# ============================================================
# 各タイムステップ描画（全タイムステップ）
# ============================================================
print("[INFO] Generating density maps for AMR data...")

# 高解像度でプロットするためのグリッド解像度
grid_resolution = 800  # 800x800グリッド（メモリ節約のため少し下げる）

# 全タイムステップを処理
for ts_idx, ts in enumerate(timesteps):
    print(f"[INFO] Processing timestep {ts} ({ts_idx+1}/{len(timesteps)})")
    
    # 全ブロックの点と密度を収集
    points_list = []
    dens_list = []
    
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            # AMRデータはセル中心の座標を取得
            points = grid.cell_centers().points
            dens = grid['dens']
            
            # z=0付近のデータのみ抽出（z_sliceの近傍）
            z_tolerance = 5.0  # 許容範囲
            mask_z = np.abs(points[:, 2]) <= z_tolerance
            points_list.append(points[mask_z])
            dens_list.append(dens[mask_z])
        except Exception as e:
            continue
    
    if not points_list:
        print(f"[WARNING] No data for timestep {ts}, skipping...")
        continue
    
    pts_all = np.vstack(points_list)
    dens_all = np.hstack(dens_list)
    
    # 最初の数ステップと一定間隔でデバッグ情報を表示
    if ts_idx < 3 or ts_idx % 20 == 0:
        print(f"[DEBUG] Timestep {ts}: {len(pts_all)} points")
        print(f"[DEBUG] X range: {pts_all[:,0].min():.1f} - {pts_all[:,0].max():.1f}")
        print(f"[DEBUG] Y range: {pts_all[:,1].min():.1f} - {pts_all[:,1].max():.1f}")
        print(f"[DEBUG] Density range: {dens_all.min():.3e} - {dens_all.max():.3e}")
    
    # ====================================================
    # 全体図用：高解像度グリッドに補間
    # ====================================================
    # 均一グリッドを作成
    xs_full = np.linspace(x_min, x_max, grid_resolution)
    ys_full = np.linspace(y_min, y_max, grid_resolution)
    X_full, Y_full = np.meshgrid(xs_full, ys_full)
    
    # グリッドポイントに補間（cubicの方が滑らかだが計算重いのでlinear）
    points_2d = np.column_stack([pts_all[:, 0], pts_all[:, 1]])
    dens_full = griddata(points_2d, dens_all, (X_full, Y_full), method='linear')
    
    # ====================================================
    # 内側領域用：より高解像度
    # ====================================================
    mask_inner = np.sqrt(pts_all[:, 0]**2 + pts_all[:, 1]**2) <= high_res_radius
    pts_inner = pts_all[mask_inner]
    dens_inner = dens_all[mask_inner]
    
    if len(pts_inner) > 100:  # 十分なデータがある場合
        inner_resolution = 600  # 内側は高解像度
        xs_inner = np.linspace(-high_res_radius, high_res_radius, inner_resolution)
        ys_inner = np.linspace(-high_res_radius, high_res_radius, inner_resolution)
        X_inner, Y_inner = np.meshgrid(xs_inner, ys_inner)
        
        points_inner_2d = np.column_stack([pts_inner[:, 0], pts_inner[:, 1]])
        dens_inner_fine = griddata(points_inner_2d, dens_inner, (X_inner, Y_inner), method='linear')
    else:
        X_inner, Y_inner, dens_inner_fine = None, None, None
    
    # ====================================================
    # プロット
    # ====================================================
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 3, figure=fig, width_ratios=[4, 4, 0.3], wspace=0.3)
    
    # 全体図
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_aspect('equal')
    
    im1 = ax1.pcolormesh(X_full, Y_full, dens_full,
                         cmap='inferno',
                         norm=LogNorm(vmin=vmin, vmax=vmax),
                         shading='auto',
                         rasterized=True)  # メモリ節約
    
    # 高解像度領域を示す四角形
    rect = patches.Rectangle(
        (-high_res_radius, -high_res_radius),
        2*high_res_radius, 2*high_res_radius,
        linewidth=2, edgecolor='cyan', facecolor='none',
        linestyle='--', label=f'Zoom region (r={high_res_radius})'
    )
    ax1.add_patch(rect)
    
    ax1.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
    
    for r in [50, 100, 150, 200]:
        if r <= Lbox/2:
            theta = np.linspace(0, 2*np.pi, 200)
            ax1.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.3, linewidth=0.8)
            if r in [100, 200]:
                ax1.text(xc + r*0.7, yc + r*0.7, f'r={r}', 
                        color='white', fontsize=8, alpha=0.6)
    
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    ax1.set_xlabel("X", fontsize=12)
    ax1.set_ylabel("Y", fontsize=12)
    ax1.set_title(f"Full Domain (448³) - AMR", fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    ax1.legend(loc='upper right', framealpha=0.7, fontsize=8)
    
    # 拡大図
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_aspect('equal')
    
    if X_inner is not None and dens_inner_fine is not None:
        im2 = ax2.pcolormesh(X_inner, Y_inner, dens_inner_fine,
                             cmap='inferno',
                             norm=LogNorm(vmin=vmin, vmax=vmax),
                             shading='auto',
                             rasterized=True)
        
        ax2.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
        
        for r in [20, 40, 60, 80, 100]:
            theta = np.linspace(0, 2*np.pi, 200)
            ax2.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.5, linewidth=0.8)
            ax2.text(xc + r*0.7, yc + r*0.7, f'r={r}', 
                    color='white', fontsize=9, alpha=0.7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
        
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
        ax2.set_xlabel("X", fontsize=12)
        ax2.set_ylabel("Y", fontsize=12)
        ax2.set_title(f"Zoomed Region (r ≤ {high_res_radius})\nAMR Data", 
                     fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
        ax2.legend(loc='upper right', framealpha=0.7, fontsize=8)
    else:
        ax2.text(0.5, 0.5, 'No data in inner region', 
                transform=ax2.transAxes, ha='center', va='center', fontsize=12)
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
    
    # カラーバー
    cax = fig.add_subplot(gs[0, 2])
    cbar = fig.colorbar(im1, cax=cax, label="Density", extend='both')
    cbar.set_label("Density", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    
    # タイトル
    fig.suptitle(f"Density Distribution (x-y plane at z=0)\n"
                 f"ρ ∝ r^-1.75 | Timestep: {ts:05d} | AMR_ON", 
                 fontsize=14, fontweight='bold', y=0.98)
    
    plt.subplots_adjust(top=0.92, bottom=0.08, left=0.05, right=0.95, wspace=0.25)
    
    # 保存
    png = os.path.join(output_dir, f"density_xy_timestep_{ts:05d}_zoom_amr.png")
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    # 進捗表示
    if (ts_idx + 1) % 10 == 0:
        print(f"[INFO] Processed {ts_idx + 1}/{len(timesteps)} timesteps")

print(f"[INFO] All AMR density maps saved to {output_dir}")
print(f"[INFO] Total images: {len(timesteps)}")